# Can We Predict Default from Client Characteristics Alone?

## Motivation

Our main analysis (see `01_credit_default_risk_segmentation.ipynb`) found that the strongest predictors of default are measures of **past payment delinquency** — features like MAX_DELAY, NUM_LATE, and PAY_0. While these produce strong model performance (AUC = 0.770), they have a practical limitation: they require months of payment history. This makes the model useful for monitoring existing clients but **unusable for assessing new clients at the point of onboarding.**

This notebook explores a complementary question: **How well can we predict default using only information available when a client first applies for a credit card?**

The features available at onboarding are:
- **LIMIT_BAL** — the credit limit assigned by the bank (based on their initial assessment)
- **SEX** — gender
- **EDUCATION** — education level
- **MARRIAGE** — marital status
- **AGE** — client age

That's only 5 features, compared to 29+ in the full model. We expect significantly weaker performance — and **that itself is the key finding.**

In [ ]:
import subprocess, sys
def install(pkg):
    try: __import__(pkg.split('[')[0].replace('-','_'))
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['ucimlrepo', 'xgboost', 'imbalanced-learn']:
    install(pkg)

import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, roc_curve, classification_report,
                             confusion_matrix, precision_recall_curve, auc)
from imblearn.over_sampling import SMOTE
import xgboost as xgb

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print('Libraries loaded.')

In [ ]:
# ============================================================
# Load and prepare data
# ============================================================
from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=350)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)
df.columns = ['LIMIT_BAL','SEX','EDUCATION','MARRIAGE','AGE',
              'PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6',
              'BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6',
              'PAY_AMT1','PAY_AMT2','PAY_AMT3','PAY_AMT4','PAY_AMT5','PAY_AMT6','DEFAULT']

df = df.drop_duplicates()

# Clean categorical variables (same as main analysis)
df['EDUCATION'] = df['EDUCATION'].replace([0, 5, 6], 4)
df['MARRIAGE'] = df['MARRIAGE'].replace(0, 3)

print(f'Dataset: {len(df):,} clients, default rate: {df["DEFAULT"].mean():.1%}')

## 1. Onboarding Features Only

We restrict the model to the 5 features that would be available at the time a client applies for a credit card. No payment history, no bill amounts, no engineered behavioral features.

In [ ]:
# ============================================================
# Define onboarding features
# ============================================================
onboarding_features = ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE']

X = df[onboarding_features]
y = df['DEFAULT']

print('Features used (onboarding only):')
for f in onboarding_features:
    corr = df[f].corr(df['DEFAULT'])
    print(f'  {f:12s}: correlation with default = {corr:+.4f}')

print(f'\nFor comparison, the top features from the full model:')
print(f'  NUM_LATE:     correlation with default = +0.3984')
print(f'  PAY_0:        correlation with default = +0.3248')
print(f'  MAX_DELAY:    correlation with default = +0.3110')
print(f'\nThe strongest onboarding feature (LIMIT_BAL at -0.15) is ~2.5x weaker')
print(f'than the weakest behavioral feature used in the full model.')

In [ ]:
# ============================================================
# Train/test split + SMOTE (same methodology as main analysis)
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)

print(f'Train: {len(X_train_smote):,} (after SMOTE), Test: {len(X_test):,}')

In [ ]:
# ============================================================
# Train three models (same types as main analysis for comparison)
# ============================================================
models = {}

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train_smote)
models['Logistic Regression'] = lr.predict_proba(X_test_scaled)[:, 1]

# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=20,
                            random_state=42, n_jobs=-1)
rf.fit(X_train_smote, y_train_smote)
models['Random Forest'] = rf.predict_proba(X_test)[:, 1]

# XGBoost
xgb_m = xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                           random_state=42, eval_metric='auc')
xgb_m.fit(X_train_smote, y_train_smote)
models['XGBoost'] = xgb_m.predict_proba(X_test)[:, 1]

# Results
print('=' * 65)
print(f'{"Model":25s} {"Onboarding AUC":>15s} {"Full Model AUC":>15s}')
print('=' * 65)

# Full model AUCs from main analysis for comparison
full_model_aucs = {
    'Logistic Regression': 0.7198,
    'Random Forest': 0.7700,
    'XGBoost': 0.7634
}

for name, proba in models.items():
    onb_auc = roc_auc_score(y_test, proba)
    full_auc = full_model_aucs[name]
    drop = full_auc - onb_auc
    print(f'{name:25s} {onb_auc:>14.4f}  {full_auc:>14.4f}   (drop: {drop:+.4f})')

In [ ]:
# ============================================================
# Visual comparison: ROC curves side by side
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = {'Logistic Regression': '#3498db', 'Random Forest': '#2ecc71', 'XGBoost': '#e74c3c'}

# Onboarding model ROC
for name, proba in models.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    onb_auc = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, color=colors[name], linewidth=2,
                 label=f'{name} (AUC={onb_auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_title('Onboarding Features Only (5 features)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Reference lines showing full model performance
axes[1].set_title('Full Behavioral Model (29+ features)\n(from main analysis)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
for name, full_auc in full_model_aucs.items():
    axes[1].annotate(f'{name}: AUC = {full_auc:.3f}',
                     xy=(0.3, 0.5 - list(full_model_aucs.keys()).index(name) * 0.08),
                     fontsize=12, color=colors[name], fontweight='bold')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].text(0.3, 0.25, 'ROC curves from main notebook\n(not re-run here)',
             fontsize=10, style='italic', color='gray')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Model Comparison: Onboarding Features vs. Full Behavioral Features',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_onboarding_roc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Lift chart for onboarding model
# ============================================================
best_name = max(models, key=lambda x: roc_auc_score(y_test, models[x]))
best_proba = models[best_name]
best_auc = roc_auc_score(y_test, best_proba)

lift_df = pd.DataFrame({'actual': y_test.values, 'proba': best_proba})
lift_df = lift_df.sort_values('proba', ascending=False)
lift_df['decile'] = pd.qcut(range(len(lift_df)), 10, labels=False) + 1

overall_rate = y_test.mean()

print(f'Best onboarding model: {best_name} (AUC={best_auc:.4f})')
print(f'\nLift Chart:')
print(f'{"Decile":>7s} {"Default Rate":>13s} {"Lift":>6s} {"Cum Capture":>12s}')
print('-' * 45)

for d in range(1, 11):
    sub = lift_df[lift_df['decile'] == d]
    rate = sub['actual'].mean()
    lift = rate / overall_rate
    cum = lift_df[lift_df['decile'] <= d]['actual'].sum() / lift_df['actual'].sum()
    print(f'  D{d:2d}     {rate:>10.1%}    {lift:>5.2f}x   {cum:>10.1%}')

print(f'\nFor comparison, full model top-decile lift: 3.07x')
print(f'Full model top-decile cumulative capture: 30.8%')

In [ ]:
# ============================================================
# Feature importance: what drives the onboarding model?
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# XGBoost feature importance
importance = pd.Series(xgb_m.feature_importances_, index=onboarding_features).sort_values()
importance.plot(kind='barh', ax=axes[0], color='#e74c3c', edgecolor='black')
axes[0].set_title('Feature Importance (XGBoost, Onboarding Model)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Importance')

# Logistic Regression coefficients (interpretable)
coefs = pd.Series(lr.coef_[0], index=onboarding_features).sort_values()
colors_lr = ['#2ecc71' if v < 0 else '#e74c3c' for v in coefs.values]
coefs.plot(kind='barh', ax=axes[1], color=colors_lr, edgecolor='black')
axes[1].set_title('Logistic Regression Coefficients (Onboarding Model)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Coefficient (positive = increases default risk)')
axes[1].axvline(x=0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig('fig_onboarding_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Logistic Regression coefficients (scaled features):')
for feat, coef in zip(onboarding_features, lr.coef_[0]):
    direction = 'increases' if coef > 0 else 'decreases'
    print(f'  {feat:12s}: {coef:+.4f} ({direction} default risk)')

In [ ]:
# ============================================================
# Default rate by onboarding risk score quartiles
# ============================================================
# Score the full dataset with the onboarding model
X_all_scaled = scaler.transform(df[onboarding_features])
df['ONBOARDING_SCORE'] = (lr.predict_proba(X_all_scaled)[:, 1] * 100).round(1)

df['RISK_QUARTILE'] = pd.qcut(df['ONBOARDING_SCORE'], 4,
                               labels=['Q1 (Lowest Risk)', 'Q2', 'Q3', 'Q4 (Highest Risk)'])

fig, ax = plt.subplots(figsize=(10, 6))
quartile_stats = df.groupby('RISK_QUARTILE', observed=True)['DEFAULT'].agg(['mean', 'count'])
colors_q = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
bars = ax.bar(range(4), quartile_stats['mean'].values * 100, color=colors_q, edgecolor='black')

for bar, (idx, row) in zip(bars, quartile_stats.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{row["mean"]:.1%}\n(n={row["count"]:,.0f})', ha='center', fontweight='bold')

ax.set_xticks(range(4))
ax.set_xticklabels(['Q1\n(Lowest Risk)', 'Q2', 'Q3', 'Q4\n(Highest Risk)'])
ax.set_title('Default Rate by Onboarding Risk Score Quartile', fontsize=13, fontweight='bold')
ax.set_ylabel('Default Rate (%)')
ax.axhline(y=df['DEFAULT'].mean()*100, color='black', linestyle='--', alpha=0.5,
           label=f'Overall: {df["DEFAULT"].mean():.1%}')
ax.legend()

plt.tight_layout()
plt.savefig('fig_onboarding_quartiles.png', dpi=150, bbox_inches='tight')
plt.show()

q1_rate = quartile_stats.iloc[0]['mean']
q4_rate = quartile_stats.iloc[3]['mean']
print(f'Separation: Q1 default rate = {q1_rate:.1%}, Q4 = {q4_rate:.1%}')
print(f'Ratio: {q4_rate/q1_rate:.1f}x difference between lowest and highest quartile')
print(f'\nFor comparison, the behavioral segmentation achieves ~5x difference (12.7% vs 60.5%)')

## Key Findings

### 1. Onboarding features alone have weak predictive power

The best onboarding model (Random Forest) achieves an AUC of **0.605** — barely above the 0.50 random baseline and far below the full behavioral model's 0.770. The AUC drop ranges from 0.13 to 0.16 across all three model types. With only 5 demographic/credit features, there simply isn't enough signal to reliably distinguish future defaulters from non-defaulters.

| Model | Onboarding AUC | Full Model AUC | Drop |
|---|---|---|---|
| Logistic Regression | 0.593 | 0.720 | -0.127 |
| Random Forest | 0.605 | 0.770 | -0.165 |
| XGBoost | 0.601 | 0.763 | -0.163 |

### 2. LIMIT_BAL is the only meaningful onboarding feature

Among the 5 features, LIMIT_BAL has the strongest correlation with default (r = -0.15) — still weak, but 4x stronger than the next feature (SEX at r = -0.04). The logistic regression confirms this: LIMIT_BAL has the largest coefficient magnitude (-0.51). This makes sense — the bank's initial credit limit reflects their own prior risk assessment. But it also means the onboarding model is largely just echoing the bank's existing judgment.

### 3. The onboarding model still provides some separation — but modest

Despite weak overall performance, the model separates clients into quartiles with default rates from **13.9% (Q1, lowest risk) to 30.3% (Q4, highest risk)** — a 2.2x ratio. This is meaningful but far weaker than the behavioral segmentation's 5x ratio (12.7% to 60.5%). The top-decile lift is only 1.62x (vs. 3.07x for the full model), and the top decile captures just 16.2% of defaults (vs. 30.8%).

### 4. This validates the need for behavioral monitoring

The performance gap between the two models is itself the most important finding: **client characteristics at onboarding are insufficient to manage credit risk.** The difference between a 0.605 AUC and a 0.770 AUC comes entirely from payment behavior data — information that only exists after the client starts using the card.

This has a direct practical implication: **the bank's highest-leverage investment is not in better onboarding screening, but in active behavioral monitoring** — particularly detecting the first missed payment, where default risk jumps from 11.7% to 29.8%.

### Practical implication

A realistic credit risk framework needs **two complementary systems:**
1. **Onboarding scorecard** (this notebook) — for initial credit limit decisions and setting review schedules. AUC of 0.605 is limited, but the 2.2x separation between Q1 and Q4 is still better than no differentiation at all.
2. **Behavioral monitoring model** (main notebook) — for ongoing risk management once payment data accumulates. AUC of 0.770 with 3.07x top-decile lift enables targeted, efficient intervention.

The transition from system 1 to system 2 happens naturally as clients build payment history — typically after 3-6 months of activity.